In [3]:
from ..PromptTemplates import *
#提示词模版结合模型调用
#加载模型
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')

model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    extra_body={"thinking":{"type":"enabled"}}
)
#加载提示词
chatTemplate=ChatPromptTemplate(
    [
        ('system','you are a english teacher,your name is {teacher_name}'),
        ('user','I am {student_name},please tranlate "{word_input}" to english'),
    ]
)
word_input=input('Enter a word: ')
prompt=chatTemplate.invoke({"teacher_name":'张三',"student_name":'hyz',"word_input":word_input})

output=model.invoke(prompt)
print(output.content)

Hello, hyz! The Chinese word "香蕉" translates to **banana** in English. Keep up the great work!


In [7]:
#使用消息模版并使用预填充部分变量

human_template=HumanMessagePromptTemplate.from_template("请解释：{task}")
sys_template=SystemMessagePromptTemplate.from_template("你是一个{role},服务目标为{target_role}")
template=ChatPromptTemplate([
    sys_template,
    human_template
    ]
)
ai_template=template.partial(
    role='AI助手,叫小灵',
    target_role='普通用户'
)
new_prompt=ai_template.invoke({'task':'你是谁？'})
print(new_prompt)

messages=[SystemMessage(content='你是一个AI助手,叫小灵,服务目标为普通用户', additional_kwargs={}, response_metadata={}), HumanMessage(content='请解释：你是谁？', additional_kwargs={}, response_metadata={})]


In [15]:


#使用消息占位符在格式化中插入消息列表
sys_template=SystemMessagePromptTemplate.from_template("你是一个美术家")
prompt_template=ChatPromptTemplate.from_messages(
    [
        sys_template,
        MessagesPlaceholder('msgs'),
        ('human','{question}')
    ]
)
prompt_value=prompt_template.invoke({'msgs':[
    HumanMessage('简要解释一下素描是什么，30字以内'),
    AIMessage('素描是单色线条和明暗表现物体形态的绘画方式。')
],
'question':'那么解释具体一点呢？'})
print(prompt_value)

messages=[SystemMessage(content='你是一个美术家', additional_kwargs={}, response_metadata={}), HumanMessage(content='简要解释一下素描是什么，30字以内', additional_kwargs={}, response_metadata={}), AIMessage(content='素描是单色线条和明暗表现物体形态的绘画方式。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='那么解释具体一点呢？', additional_kwargs={}, response_metadata={})]


In [18]:
#使用可复用模版
messages=PromptLibrary.TUTOR.format_messages(
    subject='zz',
     level='high',
     question='你是谁？'
)
print(model.invoke(messages).content)

我是你的zz导师，专注于帮你解决高中政治备考中的各种问题，包括知识点的梳理、答题技巧的指导以及考试策略的制定。如果你准备好了，我们可以开始复习或训练！
